In [ ]:
!pip install tqdm

In [ ]:
import pandas as pd
import os
from tqdm import tqdm

# Mapeamento dos arquivos com colunas específicas
arquivos_txt = {
    "equipe": {
        "path": r"C:\Users\Willgnner\Documents\Residência-TI\DBA\dba_postgresql\homework\data\equipe.txt",
        "colunas": ["id_producao", "id_pessoa", "papel"],
        "encoding": "cp1252"  
    },
    "pessoa": {
        "path": r"C:\Users\Willgnner\Documents\Residência-TI\DBA\dba_postgresql\homework\data\pessoa.txt",
        "colunas": ["id_pessoa", "nome"],
        "encoding": "cp1252"
    },
    "producao": {
        "path": r"C:\Users\Willgnner\Documents\Residência-TI\DBA\dba_postgresql\homework\data\producao.txt",
        "colunas": ["id_producao", "titulo", "ano", "tipo_id"],
        "encoding": "cp1252"
    }
}

# Pasta de saída dos arquivos .parquet
pasta_saida = r"C:\Users\Willgnner\Documents\Residência-TI\DBA\dba_postgresql\homework\data"
os.makedirs(pasta_saida, exist_ok=True)

def converter_txt_para_parquet(nome, caminho_txt, colunas, encoding):
    try:
        with open(caminho_txt, mode="r", encoding=encoding, errors="replace") as f:
            df = pd.read_csv(
                f,
                sep="##",
                header=None,
                names=colunas,
                engine="python",
                on_bad_lines='skip'
            )
        caminho_parquet = os.path.join(pasta_saida, f"{nome}.parquet")
        df.to_parquet(caminho_parquet, engine="fastparquet", index=False)
        print(f"Ok {nome}.parquet criado com {len(df)} registros (com fallback de caracteres).")
    except Exception as e:
        print(f"[ERRO] ao converter {nome}: {e}")

# Execução
print("Iniciando conversão para PARQUET...")
for nome, meta in tqdm(arquivos_txt.items(), desc="Progresso"):
    print(f"\nConvertendo {nome}.txt...")
    converter_txt_para_parquet(nome, meta["path"], meta["colunas"], meta["encoding"])

Análise dos Dados


In [ ]:
import pandas as pd
import os

# Caminho da pasta
pasta = r"C:\Users\Willgnner\Documents\Residência-TI\DBA\dba_postgresql\homework\data"

# Lê os arquivos
df_equipe = pd.read_parquet(os.path.join(pasta, "equipe.parquet"), engine="fastparquet")
df_pessoa = pd.read_parquet(os.path.join(pasta, "pessoa.parquet"), engine="fastparquet")
df_producao = pd.read_parquet(os.path.join(pasta, "producao.parquet"), engine="fastparquet")

# Função auxiliar de resumo
def resumo_df(nome, df):
    print(f"\n{'='*60}\n{nome.upper()} - {len(df)} registros\n{'='*60}")
    print("Colunas:", list(df.columns))
    print("\nTipos de dados:\n", df.dtypes)
    print("\nPrimeiras linhas:\n", df.head())
    print("\nEstatísticas gerais:\n", df.describe(include='all'))
    print("\nValores nulos por coluna:\n", df.isnull().sum())
    print("\nDuplicatas:", df.duplicated().sum())

pd.set_option("display.max_columns", None)         
pd.set_option("display.max_rows", 100)             
pd.set_option("display.max_colwidth", None)        
pd.set_option("display.expand_frame_repr", False)   

# Análise individual
resumo_df("equipe", df_equipe)
resumo_df("pessoa", df_pessoa)
resumo_df("producao", df_producao)

In [ ]:
import pandas as pd
import os

# Caminho da pasta onde estão os arquivos .parquet
pasta = r"C:\Users\Willgnner\Documents\Residência-TI\DBA\dba_postgresql\homework\data"

# Lista de arquivos a carregar
arquivos = {
    "equipe": "equipe.parquet",
    "pessoa": "pessoa.parquet",
    "producao": "producao.parquet"
}

# Leitura e exibição
for nome, arquivo in arquivos.items():
    caminho = os.path.join(pasta, arquivo)
    df = pd.read_parquet(caminho, engine="fastparquet")
    print(f"\n{'='*60}\n{nome.upper()} - 5 primeiras linhas\n{'='*60}")
    print(df.head())

Remover Ano = 0

In [ ]:
import pandas as pd
import os

# Caminho
pasta = r"C:\Users\Willgnner\Documents\Residência-TI\DBA\dba_postgresql\homework\data"
caminho_original = os.path.join(pasta, "producao.parquet")
caminho_filtrado = os.path.join(pasta, "producao_filtrado.parquet")

# Carrega o DataFrame
df = pd.read_parquet(caminho_original, engine="fastparquet")

# Mostra os tipos originais
print("Antes da limpeza, tipos:\n", df.dtypes)

# Converte para inteiro (trata strings)
df["ano"] = pd.to_numeric(df["ano"], errors="coerce").fillna(0).astype(int)

# Mostra valores únicos antes da filtragem
print("\nValores únicos antes da limpeza:\n", sorted(df["ano"].unique()))

# Filtra
df_filtrado = df[df["ano"] != 0]

# Relatório
print(f"\nRegistros antes: {len(df)}")
print(f"Registros após remover ano = 0: {len(df_filtrado)}")
print(f"Removidos: {len(df) - len(df_filtrado)}")

# Mostra valores únicos após limpeza
print("\nValores únicos após limpeza:\n", sorted(df_filtrado["ano"].unique()))

# Salva
df_filtrado.to_parquet(caminho_filtrado, index=False, engine="fastparquet")
print(f"\nArquivo salvo em: {caminho_filtrado}")

In [ ]:
print(df_filtrado["ano"].eq(0).sum())  
print(sorted(df_filtrado["ano"].unique()))  

import pandas as pd

df["ano"] = pd.to_numeric(df["ano"], errors="coerce")

tem_zero = (df["ano"] == 0).sum()
tem_nan = df["ano"].isna().sum()

print(f"→ Quantidade de registros com ano = 0: {tem_zero}")
print(f"→ Quantidade de registros com ano = NaN: {tem_nan}")

In [ ]:
import pandas as pd

# Caminho do arquivo original
caminho = r"C:\Users\Willgnner\Documents\Residência-TI\DBA\dba_postgresql\homework\data\producao.txt"

# Lendo o arquivo com codificação apropriada
df = pd.read_csv(caminho, sep="##", header=None, engine="python", encoding="latin1")

# Mostrando as colunas detectadas
print("Número de colunas detectadas:", df.shape[1])
print("Colunas:")
for i, col in enumerate(df.columns):
    print(f" - Coluna {i}: nome sugerido = 'col_{i}'")

# Exibindo as primeiras 5 linhas
print("\nPrimeiras linhas:")
print(df.head())